# Traffic Flow Capstone

In the second course module you encountered different aspects of numerical problems and solution techniques that interact with each other. You have the interaction of spatial and time discretization operating together, and the implications for stability of the numerical solution. You saw the effect of nonlinearity in the convection equation, and the interplay between diffusion and convection in Burgers' equation. This is now a broader foundation to build your knowledge of numerical techniques.

In this capstone, you will put all this to work in a new problem setting: a model of traffic flow. Picture a stretch of one-lane road that starts with light traffic followed by a short patch of higher density. As the patch moves downstream, how does it change? This traffic model will allow you to estimate minimum and average vehicle speeds and investigate how much the computed results depend on the grid and time step.

## From density to traffic flux

Let $x$ measure distance downstream and $t$ measure time. The traffic density $\rho(x,t)$ is the number of cars per kilometer of road. We model the local vehicle speed $u$ as a function of density alone. At very low density, drivers travel near the free-flow speed $u_{\max}$. At the maximum density $\rho_{\max}$, traffic is bumper to bumper and comes to a stop. Assume that the speed decreases linearly between these limits.

$$
\label{eq-traffic-vehicle-speed}
u(\rho)=u_{\max}\left(1-\frac{\rho}{\rho_{\max}}\right).
$$

The traffic flux $F$ is the number of cars passing a point per hour. Multiplying density in cars per kilometer by vehicle speed in kilometers per hour gives

$$
\label{eq-traffic-flux}
F(\rho)=\rho u(\rho)
=u_{\max}\rho\left(1-\frac{\rho}{\rho_{\max}}\right).
$$

The flux is zero on an empty road because there are no cars to pass, and zero at bumper-to-bumper density because the cars are stationary.

```{figure} ./figures/velocity_and_flux.png
:label: fig-traffic-velocity-and-flux
:alt: Vehicle speed decreases linearly to zero, while the flux rises to a peak and falls back to zero.

Vehicle speed $u$ and traffic flux $F$ as functions of density $\rho$. The flux reaches its maximum at $\rho^\star=\rho_{\max}/2$.
```

## How density changes

There are no ramps or other sources of cars along this road. The number of cars on any stretch increases when more cars enter than leave. As in [the convection lesson](./06-1d-convection.ipynb#same-equation-different-algorithms), we express this balance with a conservation equation.

$$
\label{eq-traffic-conservation}
\frac{\partial \rho}{\partial t}+\frac{\partial F(\rho)}{\partial x}=0.
$$

Where the density is smooth, the chain rule gives $\rho_t+a(\rho)\rho_x=0$, with

$$
\label{eq-traffic-characteristic-speed}
a(\rho)=\frac{dF}{d\rho}
=u_{\max}\left(1-\frac{2\rho}{\rho_{\max}}\right).
$$

Both $u$ and $a$ have units of kilometers per hour. $u$ describes how fast the cars move, while $a$, the *characteristic speed*, describes how a small change in density travels through traffic. It's the speed at which information, such as a green light or a braking car, propagates on the road. For densities below $\rho_{\max}/2$, this change travels downstream because $a>0$. We will use the conservation form in [Equation %s](#eq-traffic-conservation) to compute the evolution of the patch, whose initial density jumps at its two edges.

## Problem setup: dense traffic patch, two scenarios

You will use this model to follow a dense patch of traffic as it moves along a one-lane road. You’ll run two scenarios that differ in free-flow speed and background density, then use the computed density to estimate minimum and average vehicle speeds at selected times. You’ll start with a coarse grid and repeat the calculations with finer grids and smaller time steps. By comparing both the speed summaries and the shape of the patch, you’ll judge which features persist under refinement and which may be due to numerical smearing.

The table gives the model parameters, baseline discretization, and speeds to report for each scenario.

| Setting | Scenario A | Scenario B |
| --- | --- | --- |
| Road length | 11 km | 11 km |
| Maximum density $\rho_{\max}$ | 250 cars/km | 250 cars/km |
| Free-flow speed $u_{\max}$ | 80 km/h | 136 km/h |
| Background and left inflow density | 10 cars/km | 20 cars/km |
| Dense-patch density | 50 cars/km | 50 cars/km |
| Baseline grid and time step | 51 points, 0.001 h | 51 points, 0.001 h |
| Speeds to report | Minimum at 0 min, average at 3 min, minimum at 6 min | Minimum at 0 min, average and minimum at 3 min |

In both scenarios, the road starts at $x=0$. Set the initial density to the dense-patch value from 2.2 km up to, but excluding, 4.4 km. Use the background density everywhere else. The comparison between scenarios reflects changes in both the free-flow speed and the background density.

The baseline grid includes both endpoints, so $\Delta x=0.22\ \mathrm{km}$ and the patch occupies the Python slice `10:20`. Each time you double the number of intervals, double both slice indices to keep the patch in the same place and preserve its width.

### Boundary conditions

The initial densities in both scenarios lie below half the maximum density. We will work in this range, where information enters at the left end of the road. Hold $\rho(0,t)$ at the scenario's background density throughout the run. The right end is an outflow boundary. Its density evolves with the solution.

### Speeds to report

Use kilometers and hours in the calculation. You will evaluate the solution at the specified times (3 minutes is 0.05 h and 6 minutes is 0.10 h), and convert the vehicle speeds to meters per second for reporting.

For the minimum, use the smallest vehicle speed on the grid. For the average, take the arithmetic mean of the vehicle speeds at all grid points, including the endpoints. This is a spatial sample average. Each grid point carries the same weight regardless of the local number of cars.

## On paper — derive and check the traffic update

Before running the simulation, work through the discrete update of the density on paper. Use the four headings below to organize your notes. Keep your equations, units, and intermediate values so you can retrace your reasoning during a debrief of your work.

1. **Speeds and units.** Write down the units of $\rho$, $u$, $F$, and $a$. Differentiate $F(\rho)$ to recover $a(\rho)$. For each scenario, use the range from background density to dense-patch density to find the sign and largest value of $a$. Explain which neighbor supplies information to an updated point. Calculate the initial minimum vehicle speed in each scenario and show the conversion to meters per second.
2. **Discrete update.** Number the grid points from $0$ at the inlet to $N$ at the outlet. Starting from [Equation %s](#eq-traffic-conservation), use a forward difference in time and a backward difference of the flux. Solve for $\rho_i^{n+1}$. Mark the time level of every quantity on the right-hand side, identify which nodes you update, and write down the value you hold fixed. Use the largest characteristic speed to write the CFL restriction and check the baseline time step in both scenarios.
3. **Vehicle balance.** Multiply the change at each updated node by $\Delta x$ and add over nodes $1$ through $N$. Write out the first and last flux differences and mark the terms that cancel. Express the change in the discrete vehicle total in terms of the two remaining fluxes.
4. **One-step prediction.** Use a separate three-point example with Scenario A's speed and maximum density, $\Delta x=1\ \mathrm{km}$, $\Delta t=0.005\ \mathrm{h}$, and old densities $[10,50,20]$ cars/km from left to right. Hold the inlet at $10$ cars/km. List the three old fluxes, calculate the new density at each updated node, and use the inlet and outlet fluxes to predict whether the discrete vehicle total increases or decreases. Compare that prediction with the change calculated from your updated densities.

Keep your first attempt. When you reach the worked update and supplied code below, compare them with your notes and record any correction beside your original reasoning. Use the supplied update for the simulation.

## In your notebook — make the model plots

Use the two scenarios to put numbers on the speed and flux curves above. The shaded bands mark the range from background density to patch density for each scenario. On the speed axes, compare the solid vehicle-speed curve with the dashed characteristic-speed curve. What tells you that the density disturbance travels downstream in both calculations?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

road_length = 11.0
rho_max = 250.0
patch_density = 50.0
scenarios = {'A': (80.0, 10.0), 'B': (136.0, 20.0)}  # (free-flow speed, background density)


def vehicle_speed(rho, u_max, rho_max):
    return u_max * (1.0 - rho / rho_max)


def traffic_flux(rho, u_max, rho_max):
    return rho * vehicle_speed(rho, u_max, rho_max)


def characteristic_speed(rho, u_max, rho_max):
    return u_max * (1.0 - 2.0 * rho / rho_max)

In [ ]:
density = np.linspace(0.0, rho_max, 251)
fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True, sharey='col')

for (scenario, (u_max, background)), (ax_speed, ax_flux) in zip(scenarios.items(), axes):
    ax_speed.plot(density, vehicle_speed(density, u_max, rho_max),
                  label='Vehicle speed')
    ax_speed.plot(density, characteristic_speed(density, u_max, rho_max),
                  '--', label='Characteristic speed')
    ax_speed.axhline(0, color='0.6', linewidth=0.8)
    ax_speed.set(title=f'Scenario {scenario}', ylabel='Speed (km/h)')
    ax_flux.plot(density, traffic_flux(density, u_max, rho_max), color='tab:green')
    ax_flux.set(title=f'Scenario {scenario}', ylabel='Flux (cars/h)')
    for ax in (ax_speed, ax_flux):
        ax.axvspan(background, patch_density, color='0.5', alpha=0.17)

axes[0, 0].legend()
for ax in axes[-1]:
    ax.set_xlabel('Density (cars/km)')
fig.tight_layout()
plt.show()

### From a place on the road to a point on the curve

The model curves above answer “What speed or flux does the model predict at this density?” Their horizontal axis is density. A snapshot of the road answers “What is the density at this location?” Its horizontal axis is distance.

Start at a labeled location on the left, read its density, then use that density on the horizontal axis on the right. Locations P and R are kilometers apart, but they have the same initial density in Scenario A, so they map to the same point on the speed curve. Location Q is inside the patch and maps to a lower speed. You can read the flux curve in the same way, using the density at the location you picked.

In [ ]:
u_max_A, background_A = scenarios['A']
fig_views, (ax_road, ax_curve) = plt.subplots(1, 2, figsize=(9, 3.4))

ax_road.stairs(
    [background_A, patch_density, background_A],
    [0.0, road_length / 5, 2 * road_length / 5, road_length],
    baseline=None, color='0.35',
)
ax_road.set(
    title='1. Read density at a road location',
    xlabel='Distance along the road (km)', ylabel='Density (cars/km)',
    xlim=(0, road_length), ylim=(0, 65),
)

ax_curve.plot(density, vehicle_speed(density, u_max_A, rho_max), color='0.35')
ax_curve.set(
    title='2. Use that density on the model curve',
    xlabel='Density (cars/km)', ylabel='Vehicle speed (km/h)',
    xlim=(0, rho_max), ylim=(0, 90),
)

for label, distance, local_density, color in (
    ('P', 1.1, background_A, 'tab:blue'),
    ('Q', 3.3, patch_density, 'tab:orange'),
    ('R', 5.5, background_A, 'tab:blue'),
):
    ax_road.scatter(distance, local_density, color=color, zorder=3)
    ax_road.annotate(label, (distance, local_density),
                     xytext=(0, 7), textcoords='offset points', ha='center',
                     color=color)

for label, local_density, color in (
    ('P and R', background_A, 'tab:blue'),
    ('Q', patch_density, 'tab:orange'),
):
    speed = vehicle_speed(local_density, u_max_A, rho_max)
    ax_curve.scatter(local_density, speed, color=color, zorder=3)
    ax_curve.annotate(label, (local_density, speed),
                      xytext=(8, 5), textcoords='offset points', color=color)

fig_views.tight_layout()
plt.show()

### Compare your discrete update with the supplied step

Apply a backward difference to the flux in [Equation %s](#eq-traffic-conservation) and solve the forward difference in time for the new density.

$$
\label{eq-traffic-conservative-update}
\rho_i^{n+1}=\rho_i^n-\frac{\Delta t}{\Delta x}
\left[F(\rho_i^n)-F(\rho_{i-1}^n)\right],
\qquad i=1,\ldots,N.
$$

The two slices of the old flux array, `flux[1:]` and `flux[:-1]`, pair each updated node with its left neighbor. The first density stays fixed, and the last density is updated by the same rule as the interior. The step returns the old outlet flux minus the old inlet flux, so the time loop can accumulate the net number of cars that left.

For the chosen densities, the characteristic speed is positive and largest at the smallest density. To see why the initial range is preserved, factor the flux difference. The factor multiplying $\rho_i^n-\rho_{i-1}^n$ is $u_{\max}[1-(\rho_i^n+\rho_{i-1}^n)/\rho_{\max}]$. This is the characteristic speed evaluated at the average of the two densities. If $a_{\max}\Delta t/\Delta x\le 1$, the update is a weighted average of the two old densities, with nonnegative weights that sum to one. The fixed inlet lies in the same range, so the initial density bounds and the same CFL limit apply at each step.

In [ ]:
def traffic_step(rho, dx, dt, u_max, rho_max):
    '''Advance one conservative step with a fixed left inlet.'''
    flux = traffic_flux(rho, u_max, rho_max)
    next_rho = rho.copy()
    next_rho[1:] = rho[1:] - (dt / dx) * (flux[1:] - flux[:-1])
    return next_rho, flux[-1] - flux[0]

### Check one step against your notes

Run the three-point example from the paper exercise. Compare each updated density with your hand result. The discrete total is $\Delta x\sum_{i=1}^N\rho_i$. The balance residual adds the net outflow over the step to the change in that total. How close is it to zero?

In [ ]:
rho_test = np.array([10.0, 50.0, 20.0])
dx_test, dt_test = 1.0, 0.005
rho_next, net_outflow = traffic_step(rho_test, dx_test, dt_test, 80.0, rho_max)
change = dx_test * (rho_next[1:].sum() - rho_test[1:].sum())
print('Updated densities (cars/km):', rho_next)
print(f'Change in discrete total (cars): {change:.2f}')
print(f'Balance residual (cars): {change + dt_test * net_outflow:.2e}')

### Set up and run the baseline calculation

The initializer uses the number of intervals, $N$, so the array contains $N+1$ points. On grids with $N$ divisible by five, integer slice indices put the patch boundaries at one-fifth and two-fifths of the road length.

The driver advances to a single requested time $T>0$. It rounds $T/\Delta t_{\max}$ up to choose the number of steps and divides $T$ by that count to get the actual step. It checks the CFL number before advancing. The driver is for the positive characteristic speeds in this capstone, with the leftmost initial value held fixed. It saves the final array, the accumulated net outflow, and the step information needed to compare runs.

In [ ]:
def traffic_initial_state(N, background):
    '''Build the road grid and the initial density patch.'''
    if N < 5 or N % 5 != 0:
        raise ValueError('Use a positive interval count divisible by five.')
    x = np.linspace(0.0, road_length, N + 1)
    rho = np.full(N + 1, float(background))
    rho[N // 5:2 * N // 5] = patch_density
    return x, rho


def advance_traffic(rho0, dx, t_final, dt_max, u_max, rho_max):
    '''Advance the positive-speed traffic model to one observation time.'''
    if not (dx > 0 and t_final > 0 and dt_max > 0 and u_max > 0 and rho_max > 0):
        raise ValueError('Use positive spacings, times, speed, and maximum density.')
    rho = np.array(rho0, dtype=float, copy=True)
    if rho.ndim != 1 or rho.size < 2 or not np.all(np.isfinite(rho)):
        raise ValueError('Use a finite, one-dimensional density array with at least two points.')
    if rho.min() < 0 or rho.max() >= rho_max / 2:
        raise ValueError('Densities must be at least zero and below half the maximum density.')

    steps = int(np.ceil(t_final / dt_max))
    dt = t_final / steps
    a_max = characteristic_speed(rho.min(), u_max, rho_max)
    cfl = a_max * dt / dx
    if cfl > 1.0:
        raise ValueError('CFL number exceeds one. Choose a smaller maximum time step.')

    net_outflow = 0.0
    for _ in range(steps):
        rho, flux_difference = traffic_step(rho, dx, dt, u_max, rho_max)
        net_outflow += dt * flux_difference

    return {'rho': rho, 'dt': dt, 'steps': steps, 'cfl': cfl,
            'net_outflow': net_outflow, 'work': (rho.size - 1) * steps}

Begin with 50 intervals and the baseline maximum step of 0.001 h. Each call starts from the same initial state for its scenario. Store the returned profiles so you can compare them with finer grids later. The six numbers are baseline estimates. The refinement investigation will show how sensitive they are to the grid and step.

In [ ]:
baseline_intervals = 50
baseline_dx = road_length / baseline_intervals
baseline_dt = 0.001
observation_times = {'A': (0.05, 0.10), 'B': (0.05,)}
baseline = {}

for scenario, (u_max, background) in scenarios.items():
    x, rho0 = traffic_initial_state(baseline_intervals, background)
    baseline[scenario] = {'x': x, 'rho0': rho0, 'runs': {}}
    for time in observation_times[scenario]:
        baseline[scenario]['runs'][time] = advance_traffic(
            rho0, baseline_dx, time, baseline_dt, u_max, rho_max
        )

print(f"{'Scenario':<9} {'Time (min)':>10} {'Statistic':>10} {'Speed (m/s)':>12}")
for scenario, time, statistic in (
    ('A', 0.0, 'minimum'), ('A', 0.05, 'average'), ('A', 0.10, 'minimum'),
    ('B', 0.0, 'minimum'), ('B', 0.05, 'average'), ('B', 0.05, 'minimum'),
):
    data = baseline[scenario]
    rho = data['rho0'] if time == 0 else data['runs'][time]['rho']
    u_max, _ = scenarios[scenario]
    speed = vehicle_speed(rho, u_max, rho_max) / 3.6
    value = speed.min() if statistic == 'minimum' else speed.mean()
    print(f'{scenario:<9} {60 * time:10.0f} {statistic:>10} {value:12.3f}')